# run pilot encoding models to set parameters of pre-registrations

In [ ]:
import pandas as pd
import nibabel as nb
import os
import numpy as np
import glob
import h5py
import hcp_utils as hcp
from sklearn.random_projection import SparseRandomProjection

import numpy as np
from naturalistic_encoding.stacking_fmri import stacking_CV_fmri, stacking_fmri
from naturalistic_encoding.ridge_tools import R2
import matplotlib.pyplot as plt
import seaborn as sns

import time
#pd.set_option('display.max_rows', None)

In [ ]:
import os
def load_audio_features(stim,delay,all_layers):
    transformer = SparseRandomProjection(n_components=50)
    

    save_features_dir = f'../data/{stim}_clips_cochresnet50/'
    
    X=[]
    file = h5py.File(f'{save_features_dir}cochresnet50_activations.h5', 'r')
    for layer in all_layers:
        data = file[layer]
        X.append(  transformer.fit_transform(  np.array(data)[:(-1*delay),:]  )  )
    
    file.close()
    return(X)

def load_video_features(stim,delay,all_layers):
    transformer = SparseRandomProjection(n_components=50)
    save_path = f'../data/{stim}_frames_resnet50/'
    X=[]
    for layer in all_layers:
        X_layer=[]
        emb = np.load(f'{save_path}{layer}.npz')
        for k in list(emb.keys()):
            X_layer.append(emb[k].flatten())
        X.append(  transformer.fit_transform(  np.array(X_layer)[:(-1*delay),:]  )  )
    return(X)

def get_parcel_indices(parcels):
    #annoying indexing. 'atlas_indices' are the indices that correspond to the 'index' column of the dataframe
    #these are used to access the indices from the atlas file, because 0 is places where there is no data
    #1 would be V1
    #'indices' are the actual indices of the dataframe... these are used to access the ROIS from the already parcellated files
    #so 0 should be V1 :/
    patternR = '|'.join(['Right_' + parcel for parcel in parcels])
    patternL = '|'.join(['Left_' + parcel for parcel in parcels])
    
    # get a boolean series where True indicates a match
    matches = atlas['label'].str.contains(patternR) | atlas['label'].str.contains(patternL)

    # get the indices that match
    atlas_indices = atlas[matches]['index'].tolist()
    indices = matches[matches].index.tolist()
    parcel_names=atlas[matches]['label'].tolist()
    return(atlas_indices,indices,parcel_names)


def load_fmri_data(im_file,delay,indices):
    # load fmri data for a subject
    img = nb.load(im_file)
    img_y = img.get_fdata()
    Y=img_y[delay:,indices]
    return(Y)

def get_subject_list():
    pilot_subjects=pd.read_csv('../data/pilots_ru_dm.csv') # load pilot subjects
    
    subjects=[]
    for sub in pilot_subjects['participant_id'].astype(str):
        
        im_file = os.environ['NATURALISTIC_ENCODING_HBN_PTEMPLATE'].format(sub=sub)
        try:
            Y=load_fmri_data(im_file,delay,indices)
            subjects.append(sub)
        except:
            print(f'missing {sub}')
    print(f'loaded {len(subjects)} subjects')
    return(subjects)

def load_glasser():
    from pathlib import Path
    root = Path(os.environ.get('NATURALISTIC_ENCODING_ATLAS', '../atlases'))
    atlas_dlabel = root / 'atlas-Glasser_space-fsLR_den-91k_dseg.dlabel.nii'
    img = nb.load(str(atlas_dlabel))
    atlas_data=img.get_fdata()
    atlas_data=atlas_data[0,:]
    atlas = pd.read_csv(root / 'atlas-Glasser_dseg.tsv', sep='\t')
    return(atlas,atlas_data)

In [ ]:
get_subject_list()

In [ ]:
# load parcel info (requires NATURALISTIC_ENCODING_ATLAS)
from pathlib import Path
atlas = pd.read_csv(Path(os.environ.get('NATURALISTIC_ENCODING_ATLAS', '../atlases')) / 'atlas-Glasser_dseg.tsv', sep='\t')
print(atlas['label'])

In [ ]:
#get my parcels of interest
parcels=[
    'V1',
    'V2',
    'V3',
    'V4',
    'MT',
    'MST',
    'V4t',
    'FST',
    'FFC',
    'V8',
    'PIT',
    'VVC',
    'VMV1',
    'VMV2',
    'VMV3',
    'V3A',
    'V3B',
    'V6',
    'V6A',
    'V7',
    'IPS1',
    'IFSa',
    'IFSp',
    'IFJa',
    'IFJp',
    'FEF',
    'STSvp',
    'STSdp',
    'STSva',
    'STSda',
    'STGa',
    'STV',
    'TPOJ1',
    'TPOJ2',
    'TPOJ3',
    'A1',
    'LBelt',
    'MBelt',
    'PBelt',
    'A4',
    'TA2',
    'A5']


#auditory parcels
parcels=[
    'STSvp',
    'STSdp',
    'STSva',
    'STSda',
    'STGa',
    'STV',
    'TPOJ1',
    'TPOJ2',
    'TPOJ3',
    'A1',
    'LBelt',
    'MBelt',
    'PBelt',
    'A4',
    'TA2',
    'A5']
# 'early' auditory parcels
parcels=[
    'A1',
    'LBelt',
    'MBelt',
    'PBelt',
    'A4',
    'TA2',
    'A5']


atlas_indices,indices,parcel_names=get_parcel_indices(parcels)

In [ ]:
delay=5

### load fmri data

In [ ]:

import os
pilot_subjects = pd.read_csv('../data/pilots_ru_dm.csv')
assert 'participant_id' in pilot_subjects.columns, 'use de-identified participant_id column'
sub = str(pilot_subjects['participant_id'].iloc[0])
im_file = os.environ['NATURALISTIC_ENCODING_HBN_PTEMPLATE'].format(sub=sub)


Y=load_fmri_data(im_file,delay,indices)

In [ ]:


subjects=get_subject_list()

## load audio features

In [ ]:
#  'conv1',
#  'bn1',
#  'conv1_relu1',
#  'maxpool1',
#  'layer1',
#  'layer2',
#  'layer3',
#  'layer4',
#  'avgpool',
#  'final/signal/word_int',
#  'final/signal/speaker_int',
#  'final/noise/labels_binary_via_int']
#my layers of interst
all_layers=['input_after_preproc',
 'conv1_relu1',
 'maxpool1',
 'layer1',
 'layer2',
 'layer3',
 'layer4',
 'avgpool']
stim='DM'

X=load_audio_features(stim,delay,all_layers)
X[0].shape

In [ ]:
len(X)

In [ ]:
### Run stacking using multiple features (Xs) and Y

start_time = time.time()
r2s, stacked_r2s, _, _, _, S_average = stacking_CV_fmri(Y, X, method = 'cross_val_ridge',n_folds = 5,score_f=R2)

print(time.time() - start_time)
## simple train-test setting (without the outermost cross-validation)


In [ ]:
### Results

## r2s: voxelwise R2(predictions using only one feature, data)
print('shape of r2s is (number of features, dim_Y), that is', r2s.shape)

## stacked_r2s: voxelwise R2(stacking predictions using all features, data)
print('shape of stacked_r2s is (dim_Y, ), that is', stacked_r2s.shape)

## S_average: optimzed voxelwise stacking weights showing how different features are combined
print('shape of S_average is (dim_Y, num of features), that is', S_average.shape)



In [ ]:
ind = np.where(r2s > 0.01)
for i in range(len(ind[0])):
    row = ind[0][i]
    col = ind[1][i]
    #print(f"Positive value found at row {row}, column {col}")
    #print(f'layer {row+1}', atlas.iloc[col]['label'], r2s[row,col])

In [ ]:
dim_Y=len(indices)

dim_layers=r2s.shape[0]

plt.figure(figsize=(20,10))

bar_width = 0.1
index_0 = np.arange(dim_Y)


plt.bar(index_0, stacked_r2s, width=bar_width, label='Stacked')
for i in np.arange(dim_layers):
    plt.bar(index_0+((i+1)*bar_width), r2s[i,:], width=bar_width, label=f'{i+1}')

plt.legend()
plt.xlabel('Voxel ID')
plt.ylabel('R2')
plt.title('Prediction Performance')

## see whic delay is the best

In [ ]:
import os
pilot_subjects=pd.read_csv('../data/pilots_ru_dm.csv') # load pilot subjects

subjects=[]
for sub in pilot_subjects['participant_id'].astype(str):
    
    im_file = os.environ['NATURALISTIC_ENCODING_HBN_PTEMPLATE'].format(sub=sub)
    try:
        img = nb.load(im_file)
        Y = img.get_fdata()
        print(Y.shape)
        subjects.append(sub)
    except:
        print(f'missing {sub}')
print(f'loaded {len(subjects)} subjects')


In [ ]:
import os
parcels=[
    'A1',
    'LBelt',
    'MBelt',
    'PBelt',
    'A4',
    'TA2',
    'A5']
atlas_indices,indices,parcel_names=get_parcel_indices(parcels)

delay_results=[]
#loop through delays
for delay in np.arange(1,8):
    print(f'delay={delay}')
    all_layers=['input_after_preproc',
     'conv1_relu1',
     'maxpool1',
     'layer1',
     'layer2',
     'layer3',
     'layer4',
     'avgpool']
    
    stim='DM'

    subject_results=[]
    for sub in subjects:
        im_file = os.environ['NATURALISTIC_ENCODING_HBN_PTEMPLATE'].format(sub=sub)
        
        Y=load_fmri_data(im_file,delay,indices)
        X=load_audio_features(stim,delay,all_layers)
        X = [array[:Y.shape[0], :] for array in X]

        r2s, stacked_r2s, _, _, _, S_average = stacking_CV_fmri(Y, X, method = 'cross_val_ridge',n_folds = 5,score_f=R2)
        subject_results.append(stacked_r2s)
    delay_results.append(subject_results)

In [ ]:
np.save('../data/delay_results_file', delay_results)

In [ ]:
for delay in np.arange(len(delay_results)):
    for sub in np.arange(len(delay)):
        

In [ ]:
delay_results.shape

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

data= np.load('../data/delay_results_file.npy')


# Create a new figure for each delay
for i in range(7):
    plt.figure(figsize=(10,2))
    sns.violinplot(data=data[i].T)
    plt.xticks(np.arange(25), ['Subject'+str(j+1) for j in range(25)])
    plt.title('Violin plot of the subjects for delay ' + str(i+1))
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming 'data' is your 3D numpy array of shape (7,24,14)
data= np.load('../data/delay_results_file.npy')

# Create a new figure for each delay and each brain region
# for i in range(7): # iterate over delays
for j in range(14): # iterate over brain regions
    plt.figure(figsize=(4,6))
    sns.violinplot(data=data[:,:,j].T)
#    plt.xticks(np.arange(24), ['Subject'+str(k+1) for k in range(24)])
    plt.title(f'Violin plot of the subjects by delay for brain region {parcel_names[j]}')
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming 'data' is your 3D numpy array of shape (7,24,14)
data= np.load('../data/delay_results_file.npy')
data.shape

In [ ]:
np.mean( data,axis=1 ).shape
data_meansub=np.mean( data,axis=1 )
plt.plot( data_meansub, linewidth=0.5 )
data_meansub_mean=np.mean( data_meansub,axis=1 )

plt.plot( data_meansub_mean,color='k',linewidth=2)

#delays of 5 or 6 TRs, (~4-5s) are optimal

plt.title('Delays 1-7 TRs, by ROI (avged across subjects), grand mean in black')

In [ ]:

# Create a new figure for each delay and each brain region
# for i in range(7): # iterate over delays
for j in range(14): # iterate over brain regions
    plt.figure(figsize=(4,6))
    sns.violinplot(data=data[:,:,j].T)
#    plt.xticks(np.arange(24), ['Subject'+str(k+1) for k in range(24)])
    plt.title(f'Violin plot of the subjects by delay for brain region {parcel_names[j]}')
    plt.show()


## model FIR to see if it does better

In [ ]:
# #this is backwards...
# def model_FIR(X,delays):
# # input: a time x feature array
# # output: a time x feature(x4) array for FIR model 
#     for d in np.arange(delays):
        
#     return X_fir
# X

## try video features

In [ ]:
import os
import numpy as np
import glob

stim='DM'
delay=5
all_layers=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']
X=load_visual_features(stim,delay,all_layers)

In [ ]:
len(X)

In [ ]:
parcels=[
    'V1',
    'V2',
    'V3',
    'V4',]

atlas_indices,indices,parcel_names=get_parcel_indices(parcels)
parcel_names

In [ ]:
import os
parcels=[
    'V1',
    'V2',
    'V3',
    'V4',]

atlas_indices,indices,parcel_names=get_parcel_indices(parcels)

delay_r2s=[]
delay_stacked_r2s=[]
delay_S_average=[]

#loop through delays
for delay in np.arange(1,8):
    print(f'delay={delay}')
    all_layers=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']
    stim='DM'

    subject_r2s=[]
    subject_stacked_r2s=[]
    subject_S_average=[]
    for sub in subjects:
        im_file = os.environ['NATURALISTIC_ENCODING_HBN_PTEMPLATE'].format(sub=sub)
        
        Y=load_fmri_data(im_file,delay,indices)
        X=load_video_features(stim,delay,all_layers)
        X = [array[:Y.shape[0], :] for array in X]

        r2s, stacked_r2s, _, _, _, S_average = stacking_CV_fmri(Y, X, method = 'cross_val_ridge',n_folds = 5,score_f=R2)
        subject_r2s.append(r2s)
        subject_stacked_r2s.append(stacked_r2s)
        subject_S_average.append(S_average)
    delay_r2s.append(subject_r2s)
    delay_stacked_r2s.append(subject_stacked_r2s)
    delay_S_average.append(subject_S_average)




In [ ]:
np.mean( data,axis=1 ).shape
data_meansub=np.mean( data,axis=1 )
plt.plot( data_meansub, linewidth=0.5 )
data_meansub_mean=np.mean( data_meansub,axis=1 )

plt.plot( data_meansub_mean,color='k',linewidth=2)

#delays of 5 or 6 TRs, (~4-5s) are optimal

plt.title('Delays 1-7 TRs, by ROI (avged across subjects), grand mean in black')

In [ ]:
np.savez('../data/delay_results_video_file', delay_r2s=delay_r2s, delay_stacked_r2s=delay_stacked_r2s, delay_S_average=delay_S_average)

In [ ]:

# # Access the variables


In [ ]:
np.asanyarray(delay_stacked_r2s).shape

In [ ]:
data=np.asanyarray(delay_stacked_r2s)
np.mean( data,axis=1 ).shape
data_meansub=np.mean( data,axis=1 )
plt.plot( data_meansub, linewidth=0.5 )
data_meansub_mean=np.mean( data_meansub,axis=1 )

plt.plot( data_meansub_mean,color='k',linewidth=2)

#delays of 5 or 6 TRs, (~4-5s) are optimal

plt.title('Delays 1-7 TRs, by ROI (avged across subjects), grand mean in black')